In [1]:
!pip install -q -U transformers accelerate datasets sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 90.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 99.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.5 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


In [51]:
import torch
import transformers
import sklearn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
Transformers: 5.0.0
CUDA Available: True
GPU: Tesla T4


In [33]:


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score
)

print("Evaluation metrics imported successfully!")

Evaluation metrics imported successfully!


In [53]:
from huggingface_hub import login

login()

In [54]:
from huggingface_hub import whoami

user_info = whoami()
print(user_info["name"])

mossarrafhossainrobin


In [55]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [57]:
model_name = "google/gemma-3-4b-it"

In [58]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [8]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded successfully!


In [10]:
print("Model device:", model.device)

Model device: cuda:0


In [12]:
import pandas as pd

In [13]:
df = pd.read_csv("/kaggle/input/datasets/robinhossain231/project2/blp23_sentiment_dev.tsv",
               sep="\t" )
print("Dataset loaded successfully!")

Dataset loaded successfully!


In [14]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Shape: (3934, 3)

Columns:
['id', 'text', 'label']

First 5 rows:


,id,text,label
0,5300,নতুন মতলব আটছে । মানে সৌদি আরব ধ্বংস করার পায়ত...,Negative
1,15392,বিদেশে পড়ালেখা করছে বাংলাদেশের প্রচুর ছেলেময়ের...,Positive
2,6904,মাননীয় আপনি নিজে না বলে সস্তায় কোনো মন্ত্রী কে...,Negative
3,30790,* করোনার টিকা নিলেন বিএনপি চেয়ারপারসন বেগম খ...,Positive
4,2770,একজন প্রধানমন্ত্রীর এমন বক্তব্য জাতির জন্য লজ্...,Negative


In [15]:
sentiment_df = df[["text", "label"]].copy()

print("Shape:", sentiment_df.shape)
print("Columns:", sentiment_df.columns.tolist())

display(sentiment_df.head())

Shape: (3934, 2)
Columns: ['text', 'label']


,text,label
0,নতুন মতলব আটছে । মানে সৌদি আরব ধ্বংস করার পায়ত...,Negative
1,বিদেশে পড়ালেখা করছে বাংলাদেশের প্রচুর ছেলেময়ের...,Positive
2,মাননীয় আপনি নিজে না বলে সস্তায় কোনো মন্ত্রী কে...,Negative
3,* করোনার টিকা নিলেন বিএনপি চেয়ারপারসন বেগম খ...,Positive
4,একজন প্রধানমন্ত্রীর এমন বক্তব্য জাতির জন্য লজ্...,Negative


In [16]:
texts = sentiment_df["text"].tolist()

print("Total texts:", len(texts))
print("Sample text:")
print(texts[0])

Total texts: 3934
Sample text:
নতুন মতলব আটছে । মানে সৌদি আরব ধ্বংস করার পায়তারা শুরু করছে । 


In [17]:
gold_labels = sentiment_df["label"].tolist()

print("Sample gold label:")
print(gold_labels[0])

Sample gold label:
Negative


In [18]:
def zero_shot_prompt(text):
    return f"""
You are a text classification model.

Read the following text and determine its sentiment.

Text:
{text}

Choose exactly one label:
positive
negative
neutral

Return only the label.
"""

In [19]:
def generate_response(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [20]:
print(type(model))

<class 'transformers.models.gemma3.modeling_gemma3.Gemma3ForConditionalGeneration'>


In [21]:
text = sentiment_df["text"].iloc[0]

prompt = zero_shot_prompt(text)

response = generate_response(prompt)

print("Text:")
print(text)

print("\nGemma Prediction:")
print(response)

print("\nGold Label:")
print(sentiment_df["label"].iloc[0])

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Text:
নতুন মতলব আটছে । মানে সৌদি আরব ধ্বংস করার পায়তারা শুরু করছে । 

Gemma Prediction:
negative

Gold Label:
Negative


In [22]:
def extract_label(response):
    response = response.upper().strip()

    if "POSITIVE" in response:
        return "POSITIVE"
    elif "NEGATIVE" in response:
        return "NEGATIVE"
    elif "NEUTRAL" in response:
        return "NEUTRAL"
    else:
        return "UNKNOWN"

In [23]:
predictions = []

for i, text in enumerate(sentiment_df["text"]):

    prompt = zero_shot_prompt(text)

    response = generate_response(prompt)

    prediction = extract_label(response)

    predictions.append(prediction)

    if (i + 1) % 50 == 0:
        print(f"Processed: {i + 1}/{len(sentiment_df)}")

Processed: 50/3934
Processed: 100/3934
Processed: 150/3934
Processed: 200/3934
Processed: 250/3934
Processed: 300/3934
Processed: 350/3934
Processed: 400/3934
Processed: 450/3934
Processed: 500/3934
Processed: 550/3934
Processed: 600/3934
Processed: 650/3934
Processed: 700/3934
Processed: 750/3934
Processed: 800/3934
Processed: 850/3934
Processed: 900/3934
Processed: 950/3934
Processed: 1000/3934
Processed: 1050/3934
Processed: 1100/3934
Processed: 1150/3934
Processed: 1200/3934
Processed: 1250/3934
Processed: 1300/3934
Processed: 1350/3934
Processed: 1400/3934
Processed: 1450/3934
Processed: 1500/3934
Processed: 1550/3934
Processed: 1600/3934
Processed: 1650/3934
Processed: 1700/3934
Processed: 1750/3934
Processed: 1800/3934
Processed: 1850/3934
Processed: 1900/3934
Processed: 1950/3934
Processed: 2000/3934
Processed: 2050/3934
Processed: 2100/3934
Processed: 2150/3934
Processed: 2200/3934
Processed: 2250/3934
Processed: 2300/3934
Processed: 2350/3934
Processed: 2400/3934
Processed: 2

In [26]:
print("Number of predictions:", len(predictions))
print("Number of dataframe rows:", len(sentiment_df))

if len(predictions) != len(sentiment_df):
    raise ValueError(
        f"Length mismatch: {len(predictions)} predictions "
        f"for {len(sentiment_df)} rows."
    )


sentiment_df["predicted_label"] = predictions

print("\nPrediction column added successfully!")


print("\nPredicted Label Distribution:")
print(sentiment_df["predicted_label"].value_counts())

Number of predictions: 3934
Number of dataframe rows: 3934

Prediction column added successfully!

Predicted Label Distribution:
predicted_label
NEGATIVE    2814
POSITIVE    1020
NEUTRAL      100
Name: count, dtype: int64


In [27]:
print("Gold labels:")
print(sentiment_df["label"].unique())

Gold labels:
['Negative' 'Positive' 'Neutral']


In [28]:
text = sentiment_df["text"].iloc[0]

prompt = zero_shot_prompt(text)

raw_response = generate_response(prompt)

print("TEXT:")
print(text)

print("\nRAW GEMMA RESPONSE:")
print(repr(raw_response))

print("\nGOLD LABEL:")
print(repr(sentiment_df["label"].iloc[0]))

TEXT:
নতুন মতলব আটছে । মানে সৌদি আরব ধ্বংস করার পায়তারা শুরু করছে । 

RAW GEMMA RESPONSE:
'negative'

GOLD LABEL:
'Negative'


In [29]:
y_true = sentiment_df["label"].str.upper().str.strip()
y_pred = sentiment_df["predicted_label"].str.upper().str.strip()

print("Gold labels:")
print(y_true.value_counts())

print("\nPredicted labels:")
print(y_pred.value_counts())

Gold labels:
label
NEGATIVE    1753
POSITIVE    1388
NEUTRAL      793
Name: count, dtype: int64

Predicted labels:
predicted_label
NEGATIVE    2814
POSITIVE    1020
NEUTRAL      100
Name: count, dtype: int64


In [34]:
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

kappa = cohen_kappa_score(y_true, y_pred)

In [35]:
print("===== Gemma Zero-Shot Sentiment Analysis =====")

print(f"Accuracy      : {accuracy:.4f}")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1-Score      : {f1:.4f}")
print(f"Cohen's Kappa : {kappa:.4f}")

===== Gemma Zero-Shot Sentiment Analysis =====
Accuracy      : 0.6055
Precision     : 0.5724
Recall        : 0.6055
F1-Score      : 0.5439
Cohen's Kappa : 0.3252


In [36]:
def cot_prompt(text):
    return f"""
You are a Bengali sentiment classification expert.

Analyze the sentiment of the following Bengali text carefully.

Text:
{text}

Consider:
1. Whether the text expresses positive, negative, or neutral emotion/opinion.
2. The important words or phrases that indicate the sentiment.
3. The overall sentiment of the complete text.

Then classify the text into exactly one category:

Positive
Negative
Neutral

First reason about the sentiment internally, but in the final answer
return ONLY one label:

Positive
Negative
Neutral
"""

In [37]:
text = sentiment_df["text"].iloc[0]

prompt = cot_prompt(text)

response = generate_response(prompt)

print("Text:")
print(text)

print("\nCoT Prediction:")
print(response)

print("\nGold Label:")
print(sentiment_df["label"].iloc[0])

Text:
নতুন মতলব আটছে । মানে সৌদি আরব ধ্বংস করার পায়তারা শুরু করছে । 

CoT Prediction:
```python
Negative
```
Explanation:

Gold Label:
Negative


In [40]:
def extract_label(response):
    response = response.lower().strip()

    if "positive" in response:
        return "Positive"
    elif "negative" in response:
        return "Negative"
    elif "neutral" in response:
        return "Neutral"
    else:
        return "Unknown"

In [41]:
cot_predictions = []

for i, text in enumerate(sentiment_df["text"]):

    prompt = cot_prompt(text)

    response = generate_response(prompt)

    prediction = extract_label(response)

    cot_predictions.append(prediction)

    if (i + 1) % 50 == 0:
        print(f"Processed: {i + 1}/{len(sentiment_df)}")

Processed: 50/3934
Processed: 100/3934
Processed: 150/3934
Processed: 200/3934
Processed: 250/3934
Processed: 300/3934
Processed: 350/3934
Processed: 400/3934
Processed: 450/3934
Processed: 500/3934
Processed: 550/3934
Processed: 600/3934
Processed: 650/3934
Processed: 700/3934
Processed: 750/3934
Processed: 800/3934
Processed: 850/3934
Processed: 900/3934
Processed: 950/3934
Processed: 1000/3934
Processed: 1050/3934
Processed: 1100/3934
Processed: 1150/3934
Processed: 1200/3934
Processed: 1250/3934
Processed: 1300/3934
Processed: 1350/3934
Processed: 1400/3934
Processed: 1450/3934
Processed: 1500/3934
Processed: 1550/3934
Processed: 1600/3934
Processed: 1650/3934
Processed: 1700/3934
Processed: 1750/3934
Processed: 1800/3934
Processed: 1850/3934
Processed: 1900/3934
Processed: 1950/3934
Processed: 2000/3934
Processed: 2050/3934
Processed: 2100/3934
Processed: 2150/3934
Processed: 2200/3934
Processed: 2250/3934
Processed: 2300/3934
Processed: 2350/3934
Processed: 2400/3934
Processed: 2

In [43]:
sentiment_df["cot_prediction"] = cot_predictions

In [44]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score
)

y_true = sentiment_df["label"].str.lower().str.strip()
y_pred = sentiment_df["cot_prediction"].str.lower().str.strip()

print("===== Gemma CoT Results =====")
print(f"Accuracy      : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision     : {precision_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall        : {recall_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1-Score      : {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Cohen's Kappa : {cohen_kappa_score(y_true, y_pred):.4f}")

===== Gemma CoT Results =====
Accuracy      : 0.4520
Precision     : 0.5741
Recall        : 0.4520
F1-Score      : 0.5028
Cohen's Kappa : 0.2318


In [59]:
def few_shot_prompt(text):
    return f"""
You are a text classification model.

Learn from these examples:

Example 1:
Text: এই পণ্যটি খুব ভালো এবং আমি এটি পছন্দ করেছি।
Sentiment: positive

Example 2:
Text: সেবাটি খুব খারাপ, আমি একদম সন্তুষ্ট নই।
Sentiment: negative

Example 3:
Text: পণ্যটি মোটামুটি, বিশেষ ভালো বা খারাপ কিছু নয়।
Sentiment: neutral

Now classify this text:

Text:
{text}

Choose exactly one label:
positive
negative
neutral

Return only the label.
"""

In [60]:
def generate_response(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [61]:
predictions = []

for i, text in enumerate(sentiment_df["text"]):

    prompt = few_shot_prompt(text)

    response = generate_response(prompt)

    prediction = extract_label(response)

    predictions.append(prediction)

    if (i + 1) % 50 == 0:
        print(f"Processed: {i + 1}/{len(sentiment_df)}")

sentiment_df["predicted_label"] = predictions

print("Prediction column added successfully!")
print(sentiment_df["predicted_label"].value_counts())

Processed: 50/3934
Processed: 100/3934
Processed: 150/3934
Processed: 200/3934
Processed: 250/3934
Processed: 300/3934
Processed: 350/3934
Processed: 400/3934
Processed: 450/3934
Processed: 500/3934
Processed: 550/3934
Processed: 600/3934
Processed: 650/3934
Processed: 700/3934
Processed: 750/3934
Processed: 800/3934
Processed: 850/3934
Processed: 900/3934
Processed: 950/3934
Processed: 1000/3934
Processed: 1050/3934
Processed: 1100/3934
Processed: 1150/3934
Processed: 1200/3934
Processed: 1250/3934
Processed: 1300/3934
Processed: 1350/3934
Processed: 1400/3934
Processed: 1450/3934
Processed: 1500/3934
Processed: 1550/3934
Processed: 1600/3934
Processed: 1650/3934
Processed: 1700/3934
Processed: 1750/3934
Processed: 1800/3934
Processed: 1850/3934
Processed: 1900/3934
Processed: 1950/3934
Processed: 2000/3934
Processed: 2050/3934
Processed: 2100/3934
Processed: 2150/3934
Processed: 2200/3934
Processed: 2250/3934
Processed: 2300/3934
Processed: 2350/3934
Processed: 2400/3934
Processed: 2

In [62]:
y_true = sentiment_df["label"].str.upper().str.strip()
y_pred = sentiment_df["predicted_label"].str.upper().str.strip()

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true, y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true, y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true, y_pred,
    average="weighted",
    zero_division=0
)

kappa = cohen_kappa_score(y_true, y_pred)

print("===== Gemma Few-Shot Sentiment Analysis =====")
print(f"Accuracy      : {accuracy:.4f}")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1-Score      : {f1:.4f}")
print(f"Cohen's Kappa : {kappa:.4f}")

===== Gemma Few-Shot Sentiment Analysis =====
Accuracy      : 0.5165
Precision     : 0.6188
Recall        : 0.5165
F1-Score      : 0.5313
Cohen's Kappa : 0.2791
